In [ ]:
import pandas as pd
import numpy as np


In [ ]:
df_bus = pd.read_csv('./data/편의점_버스정류장_둔산동_결과.csv')
df = pd.read_excel('./data/편의점_최종.xlsx', engine='openpyxl')

new_cols = ["번호","주소","매장명","pop500","elder500"]

# 1) 기존 열이름(=잘못 올라간 값들)을 한 줄 데이터로 삽입
old_cols_as_row = pd.Series(df.columns, index=df.columns)
df = pd.concat([old_cols_as_row.to_frame().T, df], ignore_index=True)

# 2) 새 열이름으로 교체
df.columns = new_cols
df.head()
df_bus.head()

df_merge = pd.merge(df_bus, df, left_on='store_name', right_on='매장명', how='left')

df_merge = df_merge[['store_name', 'pop500', 'elder500', 'bus_stops_200m', 'lon', 'lat']]

for c in ['pop500', 'elder500']:
    df_merge[c] = pd.to_numeric(
        df_merge[c].astype(str).str.replace(',', '', regex=False).str.strip(),
        errors='coerce'
    )

df_merge.astype({'pop500': 'float', 'elder500': 'float', 'bus_stops_200m': 'float'})

import geopandas as gpd
from shapely.geometry import Point

# 1) 위경도 → 포인트 생성 (lon=x, lat=y 주의!)
gdf = gpd.GeoDataFrame(
    df_merge, 
    geometry=gpd.points_from_xy(df_merge["lon"], df_merge["lat"]),
    crs="EPSG:4326"   # WGS84 (위경도)
)

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df_cluster = df_merge.copy()
df_cluster['elder_ratio'] = df_cluster['elder500'] / df_cluster['pop500']
df_cluster.drop(columns=['elder500'], inplace=True)
df_cluster.head(5)

In [ ]:
df_cluster

In [ ]:
import matplotlib.pyplot as plt

# 맥북 한글 폰트 설정
plt.rc('font', family='AppleGothic')

# minus 폰트 깨짐
plt.rc('axes', unicode_minus=False)



fig, axs = plt.subplots(1, 3, figsize=(15, 5))
df_cluster['pop500'].hist(ax=axs[0], bins=20)
axs[0].set_title('500m 내 인구수', fontsize=24)
df_cluster['elder_ratio'].hist(ax=axs[1], bins=20)
axs[1].set_title('500m 내 65세 이상  비율', fontsize=24)
df_cluster['bus_stops_200m'].hist(ax=axs[2], bins=20)
axs[2].set_title('200m 내 버스정류장 수', fontsize=24)
plt.tight_layout()
plt.show()

# 표준화 후 시각화
scaler = StandardScaler()
scaled_features = scaler.fit_transform(
    df_cluster[['pop500', 'elder_ratio', 'bus_stops_200m']]
)
df_scaled = pd.DataFrame(scaled_features, columns=['pop500', 'elder_ratio', 'bus_stops_200m'])
fig, axs = plt.subplots(1, 3, figsize=(22, 5))
df_scaled['pop500'].hist(ax=axs[0], bins=20)
axs[0].set_title('500m 내 인구수', fontsize=24)
df_scaled['elder_ratio'].hist(ax=axs[1], bins=20)
axs[1].set_title('500m 내 고령자 비율', fontsize=24)
df_scaled['bus_stops_200m'].hist(ax=axs[2], bins=20)
axs[2].set_title('200m 내 버스 정류장 수', fontsize=24)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PowerTransformer, RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

scaler = StandardScaler()

X = df_cluster[['pop500','elder_ratio','bus_stops_200m']].copy()
Xz = StandardScaler().fit_transform(X)

z = np.abs(Xz)
outlier_idx = np.where((z>3).any(axis=1))[0]   # |z|>3 한 변수라도 있으면


# === 2) k=2..5에 대해 실루엣, SSE(엘보우) 계산 ===
ks = [2, 3, 4, 5]
sil_scores = []
sses = []

for k in ks:
    km = KMeans(n_clusters=k, n_init=50, random_state=42)
    labels = km.fit_predict(Xz)
    sses.append(km.inertia_)                         # 엘보우용 SSE
    sil = silhouette_score(Xz, labels)               # 실루엣
    sil_scores.append(sil)

best_k_by_sil = ks[int(np.argmax(sil_scores))]
print(f"[실루엣] 최고값 k = {best_k_by_sil}, score = {max(sil_scores):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# === 3) 시각화: (A) 실루엣 곡선 ===
axes[0].plot(ks, sil_scores, marker='o')
axes[0].set_title('Silhouette Score by k (k=2..5)')
axes[0].set_xlabel('k (number of clusters)')
axes[0].set_ylabel('Silhouette score')
axes[0].grid(True)

# === 4) 시각화: (B) 엘보우 곡선 ===
axes[1].plot(ks, sses, marker='o')
axes[1].set_title('Elbow Method (Within-Cluster SSE) for k=2..5')
axes[1].set_xlabel('k (number of clusters)')
axes[1].set_ylabel('SSE (inertia)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
df = df_cluster.copy()

pt = PowerTransformer(method="yeo-johnson", standardize=False)
df["pop500_yj"] = pt.fit_transform(df[["pop500"]])

feat = ["pop500_yj", "elder_ratio", "bus_stops_200m"]  # 정류장 반경명에 맞게
scaler = StandardScaler()
X = scaler.fit_transform(df[feat])


ks = [3, 4, 5, 6, 7, 8, 9, 10]
sil_scores = []
sses = []

for k in ks:
        km = KMeans(n_clusters=k, n_init=50, random_state=42)
        labels = km.fit_predict(X)
        sses.append(km.inertia_)                         # 엘보우용 SSE
        sil = silhouette_score(X, labels)               # 실루엣
        sil_scores.append(sil)

best_k_by_sil = ks[int(np.argmax(sil_scores))]
print(f"[실루엣] 최고값 k = {best_k_by_sil}, score = {max(sil_scores):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(20, 3))

# (1,1) 실루엣
axes[0].plot(ks, sil_scores, marker='o')
axes[0].set_title('Silhouette Score by k (2..5)', fontsize=24)
axes[0].set_xlabel('k', fontsize=20)
axes[0].set_ylabel('Silhouette score')
axes[0].grid(True)

# (1,2) 엘보우
axes[1].plot(ks, sses, marker='o')
axes[1].set_title('Elbow Method: Within-Cluster SSE (2..5)', fontsize=24)
axes[1].set_xlabel('k', fontsize=20)
axes[1].set_ylabel('SSE (inertia)')
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_samples, silhouette_score
from sklearn.cluster import KMeans
import math

def silhouette_on_ax(ax, X, labels, title=None, show_legend=False, fontsize=11):
    n_clusters = len(np.unique(labels))
    s_vals = silhouette_samples(X, labels)
    s_avg  = silhouette_score(X, labels)
    neg_ratio = (s_vals < 0).mean()

    y_lower = 10
    for c in sorted(np.unique(labels)):
        sv = np.sort(s_vals[labels == c])
        size_c = sv.shape[0]
        y_upper = y_lower + size_c
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sv, alpha=0.8, linewidth=0)
        ax.text(-0.06, y_lower + 0.5 * size_c, str(c), fontsize=fontsize-1)
        y_lower = y_upper + 10

    ax.axvline(x=s_avg, color='k', linestyle='--', linewidth=1.2)
    if title is None:
        title = f'k={n_clusters} | mean={s_avg:.3f} | neg={neg_ratio:.1%}'
    ax.set_title(title, fontsize=fontsize+1, pad=8)
    ax.set_xlabel('Silhouette', fontsize=fontsize)
    ax.set_yticks([])
    xmin = min(-0.2, s_vals.min() - 0.02)
    ax.set_xlim([xmin, 1.0])
    if show_legend:
        ax.legend(frameon=False, fontsize=fontsize-2, loc='lower right')
    ax.grid(True, linestyle=':', alpha=0.25)

def plot_silhouette_grid_for_ks(
    X, ks, n_init=50, random_state=42,
    per_plot_width=6, height=5, dpi=140, max_cols=3, fontsize=12
):
    """
    한 행에 너무 빽빽하면 자동으로 여러 행으로 래핑.
    - per_plot_width: 개별 서브플롯의 가로폭 (단위: inch)
    - height: 각 행의 높이 (inch)
    - dpi: 해상도
    - max_cols: 한 행 최대 열 수
    """
    n = len(ks)
    ncols = min(max_cols, n)
    nrows = math.ceil(n / ncols)
    figsize = (per_plot_width * ncols, height * nrows)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=dpi, squeeze=False)
    axes = axes.ravel()

    for i, k in enumerate(ks):
        ax = axes[i]
        km = KMeans(n_clusters=k, n_init=50, random_state=random_state).fit(X)
        labels = km.labels_
        silhouette_on_ax(ax, X, labels, title=f'k={k}', fontsize=fontsize)

    # 나머지 빈 축 숨기기
    for j in range(i+1, nrows*ncols):
        axes[j].axis('off')

    plt.subplots_adjust(wspace=0.25, hspace=0.35)
    plt.show()

# 사용 예시
ks = [2,3,4,5]
plot_silhouette_grid_for_ks(
    X, ks,
    per_plot_width=7,   # ▶ 더 크게: 7~8 추천
    height=5.5,         # ▶ 세로도 넉넉히
    dpi=160,            # ▶ 해상도 업
    max_cols=2,         # ▶ 1행 2열씩 두 줄로 보기 (폭 넓게)
    fontsize=13
)


In [ ]:
def plot_silhouette_grid_for_ks(
    X, ks, n_init=50, random_state=42,
    per_plot_width=6, height=5, dpi=140, max_cols=3, fontsize=12
):
    n = len(ks)
    ncols = min(max_cols, n)
    nrows = math.ceil(n / ncols)

    # ★ 슬라이드 한 장에 가로로 길게: figsize 고정 (20, 4)
    figsize = (20, 3)

    fig, axes = plt.subplots(nrows, ncols, figsize=figsize, dpi=dpi, squeeze=False)
    axes = axes.ravel()

    for i, k in enumerate(ks):
        ax = axes[i]
        km = KMeans(n_clusters=k, n_init=50, random_state=random_state).fit(X)
        labels = km.labels_
        silhouette_on_ax(ax, X, labels, title=f'k={k}', fontsize=fontsize)
        ax.set_xlabel('')  # ★ 각 축의 'Silhouette' 표시는 지우고…

    # 남는 축 숨기기
    for j in range(i+1, nrows*ncols):
        axes[j].axis('off')

    plt.subplots_adjust(wspace=0.25, hspace=0.35)
    fig.supxlabel('Silhouette', fontsize=fontsize)  # ★ 그림 전체에 한 번만 표기
    plt.tight_layout()
    plt.show()

ks = [2, 3, 4, 5]
plot_silhouette_grid_for_ks(
    X, ks,
    per_plot_width=20/len(ks),  # 무시되어도 값은 맞춰 둠
    height=4,                   # 참고용
    dpi=200,
    max_cols=len(ks),           # ★ k 개수만큼 열을 써서 1행으로
    fontsize=20
)

In [ ]:
# ------------------------------
# 1) 원자료 히스토그램 (네 코드 유지 + 제목/라벨만 정리)
# ------------------------------
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
df_cluster['pop500'].hist(ax=axs[0], bins=20)
axs[0].set_title('500m 내 총인구', fontsize=24)
axs[0].set_xlabel('값'); axs[0].set_ylabel('count')

df_cluster['elder_ratio'].hist(ax=axs[1], bins=20)
axs[1].set_title('500m 내 고령비', fontsize=24)
axs[1].set_xlabel('값')

df_cluster['bus_stops_200m'].hist(ax=axs[2], bins=20)
axs[2].set_title('200m 내 버스정류장 수', fontsize=24)
axs[2].set_xlabel('값')

plt.tight_layout()
plt.show()

# ------------------------------
# 2) 표준화 후 히스토그램 (표준화 표시·축 통일·0기준선·캡션만 추가)
# ------------------------------
scaler = StandardScaler()
scaled_features = scaler.fit_transform(
    df_cluster[['pop500', 'elder_ratio', 'bus_stops_200m']]
)
df_scaled = pd.DataFrame(
    scaled_features, columns=['pop500', 'elder_ratio', 'bus_stops_200m']
)

fig, axs = plt.subplots(1, 3, figsize=(20, 5))

# 총인구 (z)
df_scaled['pop500'].hist(ax=axs[0], bins=15)
axs[0].set_title('500m 내 총인구 (z)', fontsize=24)
axs[0].set_xlabel('z-score'); axs[0].set_ylabel('count')
axs[0].axvline(0, linestyle='--', linewidth=1); axs[0].set_xlim(-3, 3)

# 고령비 (z)
df_scaled['elder_ratio'].hist(ax=axs[1], bins=15)
axs[1].set_title('500m 내 고령비 (z)', fontsize=24)
axs[1].set_xlabel('z-score')
axs[1].axvline(0, linestyle='--', linewidth=1); axs[1].set_xlim(-3, 3)

# 정류장 수 (z)
df_scaled['bus_stops_200m'].hist(ax=axs[2], bins=15)
axs[2].set_title('200m 내 버스정류장 수 (z)', fontsize=24)
axs[2].set_xlabel('z-score')
axs[2].axvline(0, linestyle='--', linewidth=1); axs[2].set_xlim(-3, 3)

plt.tight_layout()

# 캡션(작게 1줄) — 필요 없으면 이 블록을 주석 처리
cap = (
    f"총인구 mean {df_scaled['pop500'].mean():+.2f}, sd {df_scaled['pop500'].std():.2f}  |  "
    f"고령비 mean {df_scaled['elder_ratio'].mean():+.2f}, sd {df_scaled['elder_ratio'].std():.2f}  |  "
    f"정류장수 mean {df_scaled['bus_stops_200m'].mean():+.2f}, sd {df_scaled['bus_stops_200m'].std():.2f}"
)
plt.figtext(0.5, -0.02, cap, ha='center', fontsize=10)

plt.show()

In [ ]:
km =  KMeans(n_clusters=4, n_init=50, random_state=42)
labels = km.fit_predict(X)

df['cluster'] = labels
df.head()


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from pathlib import Path
import folium

# 1) 파일 경로 지정 (여기만 바꾸세요)
shp_path = Path('./LSMD_ADM_SECT_UMD_대전/LSMD_ADM_SECT_UMD_30_202510.shp')  # 예시(윈도우)
# shp_path = Path("/Users/you/Downloads/읍면동_경계.shp")  # 예시(mac)

gdf = gpd.read_file(shp_path, encoding='euc-kr')  # 한글 인코딩 주의
dunsan = gdf[gdf['EMD_NM'] == '둔산동']

dunsan_4326 = dunsan.to_crs(epsg=4326)


b = dunsan_4326.total_bounds  # [minx, miny, maxx, maxy]
center = [(b[1]+b[3])/2, (b[0]+b[2])/2]

m = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m)
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])

m

In [ ]:
# 2) 클러스터별 색 마커 (df에 lat, lon, cluster, store_name 등 존재)
palette = ["#377eb8","#e41a1c","#4daf4a","#984ea3","#ff7f00",
           "#a65628","#f781bf","#999999"]
uniq = sorted(df["cluster"].unique())
color_map = {c: palette[i % len(palette)] for i,c in enumerate(uniq)}

for _, r in df.dropna(subset=["lat","lon"]).iterrows():
    c = r["cluster"]
    folium.CircleMarker(
        [r["lat"], r["lon"]],
        radius=6,
        color=color_map[c], fill=True, fill_color=color_map[c], fill_opacity=0.9,
        tooltip=f'{r.get("store_name","")} | Cl {c}'
    ).add_to(m)


# ② 범례 HTML 생성 & 붙이기
legend_items = "".join([
    f'''
    <div style="display:flex;align-items:center;margin:2px 0;">
      <span style="display:inline-block;width:12px;height:12px;
                   background:{color_map[c]};border:1px solid #333;margin-right:6px;"></span>
      <span style="font-size:12px;">Cluster {c}</span>
    </div>
    '''
    for c in sorted(color_map.keys())
])

                              
legend_html = f"""
<div style="
  position: fixed; 
  bottom: 20px; left: 20px; z-index: 9999;
  background: rgba(255,255,255,0.92); 
  padding: 10px 12px; border: 1px solid #aaa; border-radius: 4px;
  box-shadow: 0 1px 4px rgba(0,0,0,0.2);
">
  <div style="font-weight:600; margin-bottom:6px;">클러스터</div>
  {legend_items}
</div>
"""

from folium import Element
m.get_root().html.add_child(Element(legend_html))


folium.LayerControl().add_to(m)

m

In [ ]:
# 씨유(CU)둔산샘머리점, 세븐일레븐대전청사점, 씨유대전둔산수정점


import numpy as np, pandas as pd

dfp = df.copy()


summary = (
    dfp.groupby('cluster')
       .agg(n=('cluster','size'),
            pop_mean=('pop500','mean'),
            elder_mean=('elder_ratio','mean'),
            bus_mean=('bus_stops_200m','mean'),
            pop_med=('pop500','median'),
            elder_med=('elder_ratio','median'),
            bus_med=('bus_stops_200m','median'))
       .round(3)
)
summary



In [ ]:
df_z = df.copy()

pt = PowerTransformer(method="yeo-johnson", standardize=False)
df_z["pop500_yj"] = pt.fit_transform(df_z[["pop500"]])

df_z[['pop500_z','elder_ratio_z','bus_stops_200m_z']] = StandardScaler().fit_transform(
    df_z[['pop500_yj','elder_ratio','bus_stops_200m']]
)

df_z.head()

In [ ]:
df['cluster'].value_counts()

In [ ]:
import numpy as np
import pandas as pd

# 스케일된 점수 열: z_pop, z_elder, z_bus (이미 계산돼 있다고 가정)
# 군집 라벨: df["cluster"]

base = {
    0: np.array([0.30, 0.50, 0.20]),
    1: np.array([0.45, 0.20, 0.35]),
    2: np.array([0.20, 0.30, 0.50]),
    3: np.array([0.35, 0.35, 0.30]),
}

δ = 0.07   # 흔들기 진폭(±)
γ = 0.05   # 최고가중치 지배여유
SETS = 10  # 군집별 가중치 세트 수

def jitter_weights(w, δ=0.07, γ=0.05, sets=10, rng=42):
    rng = np.random.default_rng(rng)
    wid = np.argmax(w)  # 최고 가중치 인덱스 고정
    Ws = []
    for _ in range(sets):
        v = w.copy()
        idx = [i for i in range(3) if i != wid]
        # 두 성분만 ±δ 범위에서 랜덤 조정
        v[idx] += rng.uniform(-δ, δ, size=2)
        v = np.clip(v, 0.0, None)
        v = v / v.sum()
        # 지배여유 유지: 최고가중치가 나머지보다 γ 이상 크도록 보정
        if not (v[wid] >= max(v[i] for i in idx) + γ):
            # 부족하면 최고가중치로 조금 이동
            diff = (max(v[i] for i in idx) + γ) - v[wid]
            take = min(diff, 0.5*v[idx].sum())
            share = take / v[idx].sum() if v[idx].sum() > 0 else 0
            v[idx] *= (1 - share)
            v[wid] += take
            v = v / v.sum()
        Ws.append(v)
    return np.array(Ws)

# 군집별 보팅 점수 계산
df = df_z.copy()
rank_cols = []
for c in sorted(df["cluster"].unique()):
    Ws = jitter_weights(base[c], δ=δ, γ=γ, sets=SETS)
    sub = df["cluster"] == c
    S = []
    for w in Ws:
        s = w[0]*df["pop500_z"] + w[1]*df["elder_ratio_z"] + w[2]*df["bus_stops_200m_z"]
        S.append(s.rank(ascending=False))  # 1위가 가장 작음
    df.loc[sub, f"rank_avg_c{c}"] = pd.concat(S, axis=1)[sub].mean(axis=1)
    rank_cols.append(f"rank_avg_c{c}")

# 최종 보팅 순위(군집별 평균랭크 사용: 낮을수록 우선)
df["rank_vote"] = df[rank_cols].mean(axis=1)


In [ ]:
df_0 = df[df['cluster']==0].sort_values('rank_avg_c0').head(5)
df_1 = df[df['cluster']==1].sort_values('rank_avg_c1').head(8)
df_2 = df[df['cluster']==2].sort_values('rank_avg_c2').head(5)
df_3 = df[df['cluster']==3].sort_values('rank_avg_c3').head(10)

df_filter = pd.concat([df_0, df_1, df_2, df_3])
df_filter.reset_index(drop=True, inplace=True)
df_filter.head()

In [ ]:
df_filter

In [ ]:
m_filter = folium.Map(location=center, zoom_start=13, tiles="CartoDB positron")
folium.GeoJson(
    dunsan_4326.to_json(),
    name="둔산동 경계",
    style_function=lambda f: {"color":"#1f77b4","weight":2,"fill":False},
).add_to(m_filter)
m_filter.fit_bounds([[b[1], b[0]], [b[3], b[2]]])

# 2) 클러스터별 색 마커 (df에 lat, lon, cluster, store_name 등 존재)
palette = ["#377eb8","#e41a1c","#4daf4a","#984ea3","#ff7f00",
           "#a65628","#f781bf","#999999"]
uniq = sorted(df_filter["cluster"].unique())
color_map = {c: palette[i % len(palette)] for i,c in enumerate(uniq)}

for _, r in df_filter.dropna(subset=["lat","lon"]).iterrows():
    c = r["cluster"]
    folium.CircleMarker(
        [r["lat"], r["lon"]],
        radius=6,
        color=color_map[c], fill=True, fill_color=color_map[c], fill_opacity=0.9,
        tooltip=f'{r.get("store_name","")} | Cl {c}'
    ).add_to(m_filter)

folium.LayerControl().add_to(m_filter)

# ② 범례 HTML 생성 & 붙이기
legend_items = "".join([
    f'''
    <div style="display:flex;align-items:center;margin:2px 0;">
      <span style="display:inline-block;width:12px;height:12px;
                   bacbkground:{color_map[c]};border:1px solid #333;margin-right:6px;"></span>
      <span style="font-size:12px;">Cluster {c}</span>
    </div>
    '''
    for c in sorted(color_map.keys())
])
                                        
legend_html = f"""
<div style="
  position: fixed; 
  bottom: 20px; left: 20px; z-index: 9999;
  background: rgba(255,255,255,0.92); 
  padding: 10px 12px; border: 1px solid #aaa; border-radius: 4px;
  box-shadow: 0 1px 4px rgba(0,0,0,0.2);
">
  <div style="font-weight:600; margin-bottom:6px;">클러스터</div>
  {legend_items}
</div>
"""

from folium import Element
m_filter.get_root().html.add_child(Element(legend_html))
m_filter


In [ ]:
df2 = df[df['store_name'].isin(['씨유(CU)둔산샘머리점', '세븐일레븐대전청사점', '씨유대전둔산수정점'])]

df_f = pd.concat([df_filter, df2], axis=0)
df_f.reset_index(drop=True, inplace=True)
df_f

# 1, 2, 8, 9, 11, 15, 16, 17, 19, 22, 25, 26, 27 제외

df_f = df_f.drop(index=[1, 2, 8, 9, 11, 15, 16, 17, 19, 22, 25, 26, 27])
df_f.reset_index(drop=True, inplace=True)
df_f

In [ ]:
df_f.to_csv('./data/1차필터링.csv', index=False)
df_f
df_f[['store_name', 'rank_vote', 'lon', 'lat']]
